## MVP do nosso modelo

### Importando os dados

In [0]:
import pandas as pd
import numpy as np

In [0]:
df_train = pd.read_csv("/Workspace/Users/arthur_torres_camargo@hotmail.com/Machine-Learning/Kaggle_Competition/data/train.csv")
df_test = pd.read_csv("/Workspace/Users/arthur_torres_camargo@hotmail.com/Machine-Learning/Kaggle_Competition/data/test.csv")

In [0]:
df_train.head()

In [0]:
df_test.head()

### Criando nossa pipeline de tratamento e limpeza dos dados

In [0]:
def preparar_dados(df):
  df_clean = df.copy()

  df_clean["Embarked"] = df_clean["Embarked"].fillna('S')

  if "Fare" in df_clean.columns:
        df_clean["Fare"] = df_clean["Fare"].fillna(df_clean["Fare"].median())
  
  df_clean["Age"] = df_clean["Age"].fillna(df_clean.groupby("Pclass")["Age"].transform("median"))

  df_clean["Sex"] = df_clean["Sex"].map({"female": 0, "male": 1})

  df_clean["qtd_familia"] = df_clean["SibSp"] + df_clean["Parch"]
  df_clean["viajava_sozinho"] = df_clean["qtd_familia"].apply(lambda x: 1 if x == 0 else 0)

  df_encoded = pd.get_dummies(df_clean["Embarked"], prefix="Embarked", drop_first=True).astype(int)
  df_clean = pd.concat([df_clean, df_encoded], axis=1)

  colunas_lixo = ["PassengerId", "Name", "Ticket", "Cabin", "Embarked", "SibSp", "Parch", "qtd_familia"]
  df_clean.drop(columns=colunas_lixo, inplace=True, errors='ignore')

  return df_clean


In [0]:
train_limpo = preparar_dados(df_train)
test_limpo = preparar_dados(df_test)

In [0]:
print("Treino:", train_limpo.shape)
print("Teste:", test_limpo.shape)

### Preparando nossos dados e gerando as amostras

In [0]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [0]:
X = train_limpo.drop(columns=['Survived'])
y = train_limpo['Survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [0]:
scaler = StandardScaler()
colunas_para_escala = ['Age', 'Fare']

X_train[colunas_para_escala] = scaler.fit_transform(X_train[colunas_para_escala])
X_test[colunas_para_escala] = scaler.fit_transform(X_test[colunas_para_escala])

In [0]:
X_kaggle = test_limpo.copy()
X_kaggle[colunas_para_escala] = scaler.transform(X_kaggle[colunas_para_escala])

In [0]:
print("Tamanho do X_train (Para o modelo estudar):", X_train.shape)
print("Tamanho do X_test (Para testarmos a precisão localmente):", X_test.shape)

### Rodando o modelo

In [0]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

model = LogisticRegression(random_state=42)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

In [0]:
acuracia = accuracy_score(y_test, y_pred)

print(f"Acurácia do modelo inicial: {acuracia * 100:.2f}%")

In [0]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],          
    'penalty': ['l1', 'l2'],             
    'solver': ['liblinear', 'saga']     
}


modelo_grid = LogisticRegression(random_state=42, max_iter=2000)

grid_search = GridSearchCV(
    estimator=modelo_grid, 
    param_grid=param_grid, 
    cv=5,                                 
    scoring='accuracy',                   
    return_train_score=True,
    n_jobs=-1                           
)

grid_search.fit(X_train, y_train)

In [0]:
print(f"Melhor Acurácia na Validação Cruzada: {grid_search.best_score_ * 100:.2f}%")
print(f"Melhores Parâmetros Encontrados: {grid_search.best_params_}")

### Submissão do modelo

In [0]:
y_test_pred = grid_search.predict(X_test)
acuracia_final = accuracy_score(y_test, y_test_pred)

print(f"Acurácia Final no X_test: {acuracia_final * 100:.2f}%\n")

In [0]:
previsoes_kaggle = grid_search.predict(X_kaggle)

submissao = pd.DataFrame({
    "PassengerId": df_test["PassengerId"],
    "Survived": previsoes_kaggle
})

nome_arquivo = "/Workspace/Users/arthur_torres_camargo@hotmail.com/Machine-Learning/Kaggle_Competition/submissao_regressao_logistica_v1.csv"
submissao.to_csv(nome_arquivo, index=False)